# Loss Gamma/Tweedie
**tabular › regression_advanced** · notebook `gamma_tweedie`

**Kapan dipakai:** Target positif, miring ke kanan, dengan galat yang tumbuh seiring level (harga, klaim, durasi).

**Jebakan umum:** Perlu target > 0 (Gamma). Bandingkan dengan transformasi log; metrik evaluasi harus selaras dengan loss.

## 0. Setup
Import dan lokasi proyek. Tidak perlu diubah.

In [ ]:
import sys, pathlib, warnings
ROOT = next(p for p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]
            if (p / "src" / "superdb").is_dir())
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 50)
from superdb.common import *
from superdb.paths import out_dir

## 1. Konfigurasi
Ganti bagian ini saja saat memakai data lomba.

In [ ]:
TRAIN_PATH = ROOT / "data_latihan/tabular/regression/diamonds_train.csv"
TEST_PATH  = ROOT / "data_latihan/tabular/regression/diamonds_test.csv"      # None kalau tidak ada file test
TARGET     = "price"
ID_COL     = 'id'
DROP_COLS  = []
GROUP_COL  = None                      # isi bila ada grup agar tidak bocor
SCORING    = "neg_root_mean_squared_error"
SEED, N_SPLITS, VALID_SIZE = 42, 5, 0.2
MODEL_KEY  = "gamma_tweedie"
TASK       = "regression"
KIND       = "tree"
OUT = out_dir("tabular", "regression_advanced", MODEL_KEY)
set_seed(SEED)

## 2. Baca data

In [ ]:
df = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH) if TEST_PATH else None
print(df.shape, None if test is None else test.shape)
display(df.head(3))

## 3. Fitur & split

In [ ]:
from superdb.tabular import detect_columns, prepare_features, build_pipeline, split_train_valid, cv_score
cols = detect_columns(df, target=TARGET, id_col=ID_COL, drop_cols=DROP_COLS)
X, NUM, CAT = prepare_features(df, cols)
y = df[TARGET].to_numpy(dtype=float)
groups = df[GROUP_COL] if GROUP_COL else None
Xtr, Xva, ytr, yva = split_train_valid(X, y, TASK, VALID_SIZE, SEED, groups)
GTR = None if groups is None else groups.iloc[Xtr.index]   # grup untuk CV
print(Xtr.shape, Xva.shape)

## 4. Model dasar

In [ ]:
from lightgbm import LGBMRegressor
from sklearn.linear_model import Ridge
from sklearn.base import clone
lgbm = LGBMRegressor(n_estimators=400, learning_rate=0.05, num_leaves=31, subsample=0.8, subsample_freq=1, colsample_bytree=0.8, random_state=SEED, n_jobs=-1, verbose=-1)
def fit_eval(name, model_, kind="tree", ytr_=None, store=None):
    pp = build_pipeline(model_, NUM, CAT, kind=kind).fit(Xtr, ytr if ytr_ is None else ytr_)
    pr = pp.predict(Xva)
    if store is not None: store[name] = pr
    return {"model": name, **regression_metrics(yva, pr)}

## 5. Loss yang cocok untuk target positif miring
Gamma menganggap galat tumbuh seiring level (varian ∝ rata-rata²), sesuai harga/klaim/durasi.

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor as HGB
from sklearn.linear_model import TweedieRegressor
from sklearn.metrics import mean_gamma_deviance
assert (ytr > 0).all(), "Gamma butuh target > 0"
preds = {}
res = pd.DataFrame([
    fit_eval("HGB squared_error", HGB(random_state=SEED), store=preds),
    fit_eval("HGB gamma", HGB(loss="gamma", random_state=SEED), store=preds),
    fit_eval("LGBM l2", clone(lgbm), store=preds),
    fit_eval("LGBM gamma", LGBMRegressor(objective="gamma", n_estimators=400, learning_rate=0.05, random_state=SEED, n_jobs=-1, verbose=-1), store=preds),
    fit_eval("Ridge", Ridge(alpha=1.0), kind="linear", store=preds),
    fit_eval("GLM Gamma (log link)", TweedieRegressor(power=2, link="log", alpha=1e-3, max_iter=2000), kind="linear", store=preds)]).set_index("model")
res["gamma_deviance"] = {k: mean_gamma_deviance(yva, np.clip(v, 1e-6, None)) for k, v in preds.items()}
display(res[["rmse", "mae", "mape", "gamma_deviance"]].round(4))

In [ ]:
try:
    save_json({"method": MODEL_KEY, "valid": res.reset_index().to_dict("records") if "res" in dir() else metrics}, OUT / "metrics.json")
except NameError:
    save_json({"method": MODEL_KEY, "valid": metrics}, OUT / "metrics.json")
print("OK. hasil di", OUT)

## Langkah berikutnya
- Bandingkan metrik pada skala yang dipakai juri lomba, bukan hanya RMSE.
- Hasil terbaik biasanya digabung ke ensemble (lihat `notebooks/tabular/ensembling_regression/`).